In [53]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from opacus import PrivacyEngine

In [54]:
df = pd.read_csv("Full_data_anonymised.csv")
print("Shape:", df.shape)
df.head()

Shape: (35945, 39)


,education,urban,gender,engnat,screensize,uniquenetworklocation,hand,religion,orientation,race,...,TIPI2,TIPI3,TIPI4,TIPI5,TIPI6,TIPI7,TIPI8,TIPI9,TIPI10,Condition
0,TKN_25ff27d8,TKN_2915511a,Female,No,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,1,5,6,5,7,5,5,4,7,Extremely Severe
1,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,3,4,5,6,4,5,6,2,6,Severe
2,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_f1f2b172,TKN_9134e5db,TKN_945d5e23,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,7,6,7,6,6,6,6,3,4,Severe
3,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6311ae17,TKN_25a81701,...,5,1,7,3,4,3,7,1,2,Extremely Severe
4,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_4df44fa3,TKN_25a81701,...,2,5,5,5,5,4,2,2,2,Severe


In [55]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import torch

df_test = pd.read_csv("Test_set_Depression1_anonymised.csv")  # or test_2
# Step 2: Encode categorical columns (use same logic as training)
df_encoded = df_test.copy()
label_encoders = {}
target_column = 'Condition'

for col in df_encoded.select_dtypes(include='object').columns:
    if col != target_column:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

# Step 3: Encode the target separately
le_target = LabelEncoder()
df_encoded[target_column] = le_target.fit_transform(df_encoded[target_column])

# Step 4: Prepare X and y for evaluation
X_test = df_encoded.drop(columns=[target_column])
y_test = df_encoded[target_column]

# Step 5: Convert to PyTorch tensors
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)


In [56]:
# !pip install diffprivlib
# !pip install imbalanced-learn

In [57]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [58]:
input_dim = X_test.shape[1]  # or same as training
num_classes = 5

model = MLP(input_dim, num_classes)
model.load_state_dict(torch.load("trained_global_model_3.pth"))
model.eval()


MLP(
  (model): Sequential(
    (0): Linear(in_features=38, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)

In [59]:
# Predict on test set
with torch.no_grad():
    y_test_pred = model(X_test_tensor).argmax(dim=1)

# Evaluate
test_acc = accuracy_score(y_test_tensor, y_test_pred)
print(f"✅ Test Accuracy: {test_acc:.4f}\n")

print("📊 Classification Report (Test Set):")
print(classification_report(y_test_tensor, y_test_pred, target_names=[str(c) for c in le_target.classes_]))


✅ Test Accuracy: 0.8123

📊 Classification Report (Test Set):
                  precision    recall  f1-score   support

Extremely Severe       0.96      0.85      0.90     12432
            Mild       0.56      0.80      0.66      3546
        Moderate       0.76      0.71      0.73      6573
          Normal       0.95      0.89      0.92      8153
          Severe       0.65      0.75      0.69      6020

        accuracy                           0.81     36724
       macro avg       0.78      0.80      0.78     36724
    weighted avg       0.83      0.81      0.82     36724



In [60]:
# import torch
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# # STEP 1: Predict using the trained MLP model
# model.eval()
# with torch.no_grad():
#     y_val_logits = model(X_val_tensor)
#     y_val_pred = torch.argmax(y_val_logits, dim=1).cpu().numpy()

# # STEP 2: Optional - decode integer labels to original class names if you used LabelEncoder
# class_labels = [str(cls) for cls in le_target.classes_]  # Converts int64 to string labels

# # STEP 3: Generate the confusion matrix
# cm = confusion_matrix(y_val, y_val_pred)

# # STEP 4: Plot the confusion matrix
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
# fig, ax = plt.subplots(figsize=(8, 6))
# disp.plot(ax=ax, cmap='Blues', values_format='d')
# plt.title("Confusion Matrix - MLP Validation Set")
# plt.show()
